### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="real_estate_valuation",
    dataset_year="2013",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5J30W",
    download_description="""
wget https://archive.ics.uci.edu/static/public/477/real+estate+valuation+data+set.zip && unzip real+estate+valuation+data+set.zip && rm real+estate+valuation+data+set.zip && mkdir -p local-data-warehouse/real_estate_valuation && mv "Real estate valuation data set.xlsx" local-data-warehouse/real_estate_valuation/
""",
    # References
    academic_reference_bibtex="""@article{yeh2018building,
  title={Building real estate valuation models with comparative approach through case-based reasoning},
  author={Yeh, I-Cheng and Hsu, Tzu-Kuang},
  journal={Applied Soft Computing},
  volume={65},
  pages={260--271},
  year={2018},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="yeh2018building",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We load the data from UCI.

- We encode the transaction date as a datetime.
- We log scale the target variable.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_house_price_per_unit_area",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="transaction_datetime",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel(dataset_mold.path / "Real estate valuation data set.xlsx")
print("Loaded data shape:", df.shape)

year = df["X1 transaction date"].astype(int)
fraction = df["X1 transaction date"] - year

month = (fraction * 12 + 1).clip(1, 12).astype(int)

df["transaction_datetime"] = pd.to_datetime({
    "year": year,
    "month": month,
    "day": 1
})

df = df.drop(columns=["X1 transaction date", "No"])
df = df.rename(columns={
    "X2 house age": "house_age",
    "X3 distance to the nearest MRT station": "distance_to_nearest_MRT",
    "X4 number of convenience stores": "num_convenience_stores",
    "X5 latitude": "latitude",
    "X6 longitude": "longitude",
    "Y house price of unit area": "log_house_price_per_unit_area",
})

df["log_house_price_per_unit_area"] = np.log(df["log_house_price_per_unit_area"])

df = df.sort_values(by="transaction_datetime").reset_index(drop=True)

Loaded data shape: (414, 8)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 414
Columns: 7
Use sampling: False (sample size: 414)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['distance_to_nearest_MRT', 'house_age', 'latitude', 'longitude', 'num_convenience_stores', 'transaction_datetime']
Rows remaining as candidates after top-6 filter: 50 (of 414)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 26 (6.28% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,house_age,distance_to_nearest_MRT,num_convenience_stores,latitude,longitude,log_house_price_per_unit_area,transaction_datetime
0,30.4,1735.5950,2,24.96464,121.51623,3.254243,2012-09-01
1,37.1,918.6357,1,24.97198,121.55063,3.462606,2012-09-01
2,3.1,577.9615,6,24.97201,121.54722,3.864931,2012-09-01
3,15.6,289.3248,5,24.98203,121.54348,3.830813,2012-09-01
4,12.6,383.2805,7,24.96735,121.54464,3.749504,2012-09-01


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,transaction_datetime,datetime64[ns],0.0,0.0,8.0,"2013-01-01 00:00:00, 2013-07-01 00:00:00, 2013-04-01 00:00:00, 2012-10-01 00:00:00, 2013-06-01 00:00:00, 2012-12-01 00:00:00, 2012-09-01 00:00:00, 2013-03-01 00:00:00"
1,house_age,float64,0.0,0.0,236.0,"0.0, 13.6, 16.2, 13.3, 13.2, 16.4, 16.9, 1.1, 18.0, 4.0"
2,distance_to_nearest_MRT,float64,0.0,0.0,259.0,"289.3248, 90.4561, 492.2313, 1360.139, 104.8101, 4082.015, 390.5684, 2147.376, 4066.587, 193.5845"
3,latitude,float64,0.0,0.0,234.0,"24.9743, 24.982, 24.9667, 24.9652, 24.952, 24.963, 24.9415, 24.9794, 24.943, 24.9774"
4,longitude,float64,0.0,0.0,232.0,"121.5435, 121.5431, 121.5374, 121.5439, 121.5484, 121.5407, 121.5425, 121.5038, 121.5034, 121.5446"
5,log_house_price_per_unit_area,float64,0.0,0.0,270.0,"3.7495, 3.6964, 3.2068, 3.4436, 3.6217, 3.6243, 3.7377, 3.7448, 3.7038, 3.3776"
6,num_convenience_stores,int64,0.0,0.0,11.0,"5, 0, 1, 3, 6, 4, 7, 8, 9, 2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
house_age,414.0,17.712560,11.392485,0.000000,43.800000
distance_to_nearest_MRT,414.0,1083.885689,1262.109595,23.382840,6488.021000
num_convenience_stores,414.0,4.094203,2.945562,0.000000,10.000000
latitude,414.0,24.969030,0.012410,24.932070,25.014590
longitude,414.0,121.533361,0.015347,121.473530,121.566270
log_house_price_per_unit_area,414.0,3.566695,0.392472,2.028148,4.766438


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column               rank                                   
transaction_datetime 1     2013-01-01 00:00:00     74  17.87
                     2     2013-07-01 00:00:00     70  16.91
                     3     2013-04-01 00:00:00     61  14.73
                     4     2012-10-01 00:00:00     58  14.01
                     5     2013-06-01 00:00:00     58  14.01

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.706,-1.094,0.154,0.014,log,2278.2,2.327522e+16,exponential


## Task Curation

In [9]:
table = (
    df.groupby(task_mold.time_on)["log_house_price_per_unit_area"]
    .agg(count="size", avg_price=lambda s: s.mean())
    .reset_index()
)
table

,transaction_datetime,count,avg_price
0,2012-09-01,30,3.599145
1,2012-10-01,58,3.498747
2,2012-12-01,38,3.500700
3,2013-01-01,74,3.537620
4,2013-03-01,25,3.593873
5,2013-04-01,61,3.635732
6,2013-06-01,58,3.600286
7,2013-07-01,70,3.577949


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata

months = sorted(df[task_mold.time_on].dropna().unique())
start_test_month = 3
splits = {}

for i in range(start_test_month, len(months)):
    test_month = months[i]
    train_months = months[:i]

    train_idx = list(df.index[df[task_mold.time_on].isin(train_months)])
    test_idx = list(df.index[df[task_mold.time_on].eq(test_month)])

    splits[i - start_test_month] = {
        0: (train_idx, test_idx)
    }
    print(test_month, len(train_idx), len(test_idx))

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We simulate a model that is refit every month. We use all data from year 2012 (3 months) as base training data. Then, we take each month in 2013 (5 months) as test data once and all prior data as training data.",
    splits=splits,
    time_horizon=3,
    time_horizon_unit="months",
)

2013-01-01 00:00:00 126 74
2013-03-01 00:00:00 200 25
2013-04-01 00:00:00 225 61
2013-06-01 00:00:00 286 58
2013-07-01 00:00:00 344 70


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to real_estate_valuation/019d30a8-1b61-74d7-965e-ea11cb5e3b00
019d30a8-1b61-74d7-965e-ea11cb5e3b00
a204436429de9163009f95b0582782ea3df00cfaf6e556706796eb13f356b70e
